# Facilitator-Member Difference Identification

## Creating the csv file for the following:

`Name-conference-year`

**Also for the following:**

- facilitator vs. non-facilitator

- ppl-team vs. ppl-not-team

- ppl-funded-team vs. ppl-not-funded-team

In [2]:
import json, pandas as pd
from pathlib import Path
from collections import defaultdict, Counter

# ---- CONFIG ----
DATA_DIR = Path("/Users/maxchalekson/Desktop/gemini_data_analysis/data")   # root with 2021MZT, 2022SLU, ...
OUTPUT_DIR = Path("/Users/maxchalekson/Desktop/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ALL_PERSON_SESSION = []   # collect across all conferences
ALL_PERSON_YEAR    = []   # collect across all conferences

# ---- LOADERS ----
def _load_json(fp: Path):
    with open(fp, "r") as f:
        return json.load(f)

def load_conference_data(conf_path: Path):
    """Load session outcomes, person-to-team, and session features_* for a given conference folder."""
    conf_name   = conf_path.name   # e.g., "2021MZT"
    year        = int(conf_name[:4])
    conference  = conf_name[4:]

    outcome          = _load_json(conf_path / f"{conf_name}_outcome.json")
    person_to_team   = _load_json(conf_path / f"{conf_name}_person_to_team.json")
    session_outcomes = _load_json(conf_path / f"{conf_name}_session_outcomes.json")

    # features_*.json (per session)
    features = {}
    for fp in conf_path.glob("features_*.json"):
        sid = fp.stem.replace("features_", "")      # e.g., "2021_09_30_MZT_S5"
        features[sid] = _load_json(fp)
    return year, conference, outcome, person_to_team, session_outcomes, features

# ---- PERSON METRICS FROM TRANSCRIPTS (session_data/*.json) ----
def extract_person_metrics_from_session_data(conf_path: Path, session_id: str):
    """
    Look for /session_data/<session_id>.json and compute person-level metrics for this session.
    Returns: dict person_name -> {p_* metrics}
    """
    session_file = conf_path / "session_data" / f"{session_id}.json"
    if not session_file.exists():
        return {}

    data = _load_json(session_file)
    rows = data.get("all_data", [])
    by_person = defaultdict(lambda: Counter())

    for r in rows:
        speaker = r.get("speaker")
        if not speaker:
            continue
        dur = r.get("speaking_duration", 0) or 0
        by_person[speaker]["p_speaking_duration_sec"] += float(dur)
        by_person[speaker]["p_turns"] += 1

        if str(r.get("interuption", "")).strip().lower() == "yes":
            by_person[speaker]["p_interruptions_made"] += 1
        if str(r.get("overlap", "")).strip().lower() == "yes":
            by_person[speaker]["p_overlaps"] += 1
        if str(r.get("screenshare", "")).strip().lower() == "yes":
            by_person[speaker]["p_screenshare_segments"] += 1

        by_person[speaker]["p_smile_self_total"]  += float(r.get("smile_self", 0) or 0)
        by_person[speaker]["p_smile_other_total"] += float(r.get("smile_other", 0) or 0)
        by_person[speaker]["p_nods_received"]     += float(r.get("nods_others", 0) or 0)

    return {person: dict(cnt) for person, cnt in by_person.items()}

# ---- BUILDERS ----
def build_person_session(year, conference, conf_path, session_outcomes, features):
    """One row per (person, session), with role_in_session, ctx_* from features, and p_* from transcripts."""
    special = {"missing_names", "people_not_in_any_team"}
    sessions = [sid for sid in session_outcomes.keys() if sid not in special]
    rows = []

    for sid in sessions:
        so = session_outcomes[sid]
        facilitators = set(so.get("facilitators", []) or [])
        speakers     = set(so.get("all_speakers", []) or [])

        members = set()
        for _, tinfo in (so.get("teams", {}) or {}).items():
            for m in tinfo.get("members", []) or []:
                members.add(m)

        people = facilitators | speakers | members

        # session-level features (context)
        ctx = {f"ctx_{k}": v for k, v in (features.get(sid, {}) or {}).items()}

        # person-level from transcripts (if available)
        person_metrics = extract_person_metrics_from_session_data(conf_path, sid)

        for person in sorted(people):
            if person in facilitators:
                role_in_session = "facilitator"
            elif person in members:
                role_in_session = "member"
            elif person in speakers:
                role_in_session = "participant"
            else:
                role_in_session = "unknown"

            pmet = person_metrics.get(person, {})
            rows.append({
                "person_name": person,
                "conference": conference,
                "year": year,
                "session_id": sid,
                "role_in_session": role_in_session,
                **ctx,
                **pmet
            })

    df = pd.DataFrame(rows)
    # Optional: add session-grain flags for quick contrasts
    if not df.empty:
        df["is_facilitator"]    = (df["role_in_session"] == "facilitator").astype(int)
        df["is_member"]         = (df["role_in_session"] == "member").astype(int)
        df["is_participant"]    = (df["role_in_session"] == "participant").astype(int)
        df["is_nonfacilitator"] = (df["role_in_session"] != "facilitator").astype(int)
    return df

def build_person_year(person_session_df, person_to_team):
    """Aggregate to one row per (person, conference, year): role tallies, team counts, mean ctx_* and mean p_*."""
    if person_session_df.empty:
        return person_session_df

    pivot = (person_session_df
             .pivot_table(index=["person_name","conference","year"],
                          columns="role_in_session",
                          values="session_id",
                          aggfunc="nunique",
                          fill_value=0)
             .reset_index())

    for col in ["facilitator","member","participant","unknown"]:
        if col not in pivot.columns:
            pivot[col] = 0
    pivot["sessions_total"] = pivot["facilitator"] + pivot["member"] + pivot["participant"] + pivot["unknown"]

    role_priority = {"facilitator": 3, "member": 2, "participant": 1, "unknown": 0}
    def primary_role(row):
        return max(role_priority, key=lambda r: (row.get(r,0)>0, role_priority[r]))
    pivot["role_primary"] = pivot.apply(primary_role, axis=1)

    metric_cols = [c for c in person_session_df.columns if c.startswith("ctx_") or c.startswith("p_")]
    if metric_cols:
        agg = (person_session_df
               .groupby(["person_name","conference","year"], as_index=False)[metric_cols]
               .mean())
        out = pivot.merge(agg, on=["person_name","conference","year"], how="left")
    else:
        out = pivot

    # team outcomes
    team_rows = []
    for pname in out["person_name"]:
        lst = person_to_team.get(pname, [])
        funded = sum(1 for t in lst if t.get("funded_status", 0) == 1)
        unfund = sum(1 for t in lst if t.get("funded_status", 0) == 0)
        team_rows.append((pname, funded, unfund, ", ".join([t.get("team_id","") for t in lst])))
    team_df = pd.DataFrame(team_rows, columns=["person_name","teams_funded","teams_unfunded","team_ids"])
    team_df["teams_total"] = team_df["teams_funded"] + team_df["teams_unfunded"]

    out = out.merge(team_df, on="person_name", how="left")
    return out

# ---- DRIVER: build & COMBINE EVERYTHING INTO ONE FILE ----
for conf_path in sorted(DATA_DIR.iterdir()):
    if not conf_path.is_dir():
        continue
    conf_name = conf_path.name
    if not (conf_path / f"{conf_name}_session_outcomes.json").exists():
        continue

    year, conference, outcome, person_to_team, session_outcomes, features = load_conference_data(conf_path)
    ps = build_person_session(year, conference, conf_path, session_outcomes, features)
    py = build_person_year(ps, person_to_team)

    if not ps.empty:
        ALL_PERSON_SESSION.append(ps)
    if not py.empty:
        ALL_PERSON_YEAR.append(py)

# Combine & write ONE spreadsheet for person-year
if ALL_PERSON_YEAR:
    all_py = pd.concat(ALL_PERSON_YEAR, ignore_index=True).sort_values(
        ["person_name","year","conference"]
    )

    # --- Add role flags for multiple contrasts ---
    # 1) Facilitator vs Non-facilitator
    all_py["role_facilitator"]     = (all_py["role_primary"] == "facilitator").astype(int)
    all_py["role_nonfacilitator"]  = 1 - all_py["role_facilitator"]

    # 2) On a team vs Not on a team
    all_py["teams_total"]  = all_py["teams_total"].fillna(0)
    all_py["role_on_team"] = (all_py["teams_total"] > 0).astype(int)

    # 3) In funded team vs Not in funded team
    all_py["teams_funded"]   = all_py["teams_funded"].fillna(0)
    all_py["role_in_funded"] = (all_py["teams_funded"] > 0).astype(int)

    # Optional: pure member/participant flags
    all_py["role_member"]      = (all_py["role_primary"] == "member").astype(int)
    all_py["role_participant"] = (all_py["role_primary"] == "participant").astype(int)

    out_path = OUTPUT_DIR / "ALL_person_year.csv"
    all_py.to_csv(out_path, index=False)
    print(f"Wrote ONE combined file: {out_path}  ({len(all_py)} rows)")
else:
    print("No person-year rows produced.")

# Optional: also write ONE combined person-session file
# if ALL_PERSON_SESSION:
#     all_ps = pd.concat(ALL_PERSON_SESSION, ignore_index=True).sort_values(
#         ["person_name","year","conference","session_id"]
#     )
#     out_path_ps = OUTPUT_DIR / "ALL_person_session.csv"
#     all_ps.to_csv(out_path_ps, index=False)
#     print(f"Wrote combined person-session file: {out_path_ps}  ({len(all_ps)} rows)")

Wrote ONE combined file: /Users/maxchalekson/Desktop/outputs/ALL_person_year.csv  (790 rows)


## Regression Analysis

Figuring out the question: 

**By analyzing at the individual level, do facilitators act differently than team members?**

Also, considering again, the classification from above:

- facilitator vs. non-facilitator

- ppl-team vs. ppl-not-team

- ppl-funded-team vs. ppl-not-funded-team

In [7]:
# --- Regression sweep over person-year CSV ---
import pandas as pd
import numpy as np
from pathlib import Path
import statsmodels.formula.api as smf

# -------- CONFIG --------
CSV_PATH   = Path("/Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/ALL_person_year.csv")
OUT_DIR    = Path("/Users/maxchalekson/Desktop/outputs")
USE_CONTROLS = True   # set False if you want raw unadjusted comparisons first
CONTROL_BASICS = ["sessions_total"]   # add any other scalar controls you want here

# -------- LOAD --------
df = pd.read_csv(CSV_PATH)

# Keep only rows with non-missing role info
role_flags = ["role_facilitator", "role_on_team", "role_in_funded"]
for c in role_flags:
    if c not in df.columns:
        raise ValueError(f"Missing required role flag column: {c}")

# Auto-detect outcomes: all p_* columns that are numeric and not all-NA
p_cols = [c for c in df.columns if c.startswith("p_")]
p_cols = [c for c in p_cols if pd.api.types.is_numeric_dtype(df[c]) and df[c].notna().any()]
if not p_cols:
    raise ValueError("No p_* outcomes found in the CSV.")

# Optional controls: sessions_total + any ctx_* (if present)
ctx_cols = [c for c in df.columns if c.startswith("ctx_") and pd.api.types.is_numeric_dtype(df[c])]
controls = []
if USE_CONTROLS:
    controls = [c for c in CONTROL_BASICS if c in df.columns] + ctx_cols

def build_formula(y, x, controls_list):
    rhs = [x] + controls_list
    # dedupe & keep order
    seen = set()
    rhs_clean = []
    for v in rhs:
        if v and (v not in seen):
            rhs_clean.append(v); seen.add(v)
    if rhs_clean:
        return f"{y} ~ " + " + ".join(rhs_clean)
    else:
        return f"{y} ~ 1"

# -------- RUN REGRESSIONS --------
records = []
full_summaries = []

def subset_for_contrast(df, contrast):
    if contrast == "role_facilitator":
        # whole dataset
        return df.copy()
    elif contrast == "role_on_team":
        # compare engaged on-team vs engaged not-on-team
        d = df.copy()
        if "sessions_total" in d.columns:
            d = d[(d["sessions_total"].fillna(0) > 0)]
        return d
    elif contrast == "role_in_funded":
        # only among team members
        d = df.copy()
        if "role_on_team" not in d.columns:
            raise ValueError("role_on_team flag missing; needed for funded vs not-funded contrast.")
        return d[d["role_on_team"] == 1]
    else:
        return df.copy()

for x in ["role_facilitator", "role_on_team", "role_in_funded"]:
    df_c = subset_for_contrast(df, x)

    for y in p_cols:
        # build formula with optional controls
        needed = [y, x] + controls
        df_run = df_c[needed].dropna()
        n = len(df_run)
        if n < 20:
            continue

        formula = build_formula(y, x, controls)

        # OLS with robust (HC3) SEs
        model = smf.ols(formula, data=df_run).fit()
        robust = model.get_robustcov_results(cov_type="HC3")

        # SAFELY map coefficients by name
        names  = robust.model.exog_names
        params = dict(zip(names, robust.params))
        ses    = dict(zip(names, robust.bse))
        tvals  = dict(zip(names, robust.tvalues))
        pvals  = dict(zip(names, robust.pvalues))

        coef = params.get(x, np.nan)
        se   = ses.get(x, np.nan)
        tval = tvals.get(x, np.nan)
        pval = pvals.get(x, np.nan)

        records.append({
            "outcome": y,
            "contrast": x,
            "n_used": n,
            "controls_used": ", ".join(controls) if controls else "(none)",
            "coef_role": coef,
            "se_role": se,
            "t_role": tval,
            "p_role": pval,
            "r2": robust.rsquared,
            "r2_adj": robust.rsquared_adj
        })

        full_summaries.append({
            "outcome": y,
            "contrast": x,
            "formula": formula,
            "summary": robust.summary().as_text()
        })

# -------- SAVE RESULTS --------
OUT_DIR.mkdir(parents=True, exist_ok=True)
res_df = pd.DataFrame.from_records(records).sort_values(["contrast", "outcome"])
res_csv = OUT_DIR / "reg_results_person_year_tidy.csv"
res_df.to_csv(res_csv, index=False)

# Also save a wide pivot (outcomes x contrasts with coef & p)
coef_piv = res_df.pivot_table(index="outcome", columns="contrast", values="coef_role")
p_piv    = res_df.pivot_table(index="outcome", columns="contrast", values="p_role")
coef_piv.to_csv(OUT_DIR / "reg_results_coefs_wide.csv")
p_piv.to_csv(OUT_DIR / "reg_results_pvals_wide.csv")

# (Optional) write plain-text summaries per model
summ_path = OUT_DIR / "reg_model_summaries.txt"
with open(summ_path, "w") as f:
    for s in full_summaries:
        f.write(f"=== {s['outcome']} ~ {s['contrast']} ===\n")
        f.write(f"Formula: {s['formula']}\n")
        f.write(s["summary"])
        f.write("\n\n")

print("Wrote:")
print(f" - {res_csv}")
print(f" - {OUT_DIR / 'reg_results_coefs_wide.csv'}")
print(f" - {OUT_DIR / 'reg_results_pvals_wide.csv'}")
print(f" - {summ_path}")

# -------- QUICK HEADS-UP --------
# • 'coef_role' > 0 means the role group (e.g., facilitators) has higher outcome on average.
# • Robust SEs (HC3) used for more reliable p-values under heteroskedasticity.
# • If you later add ctx_* columns to your CSV, they will be auto-included as controls.
# • To run “raw” comparisons (no controls), set USE_CONTROLS = False.

Wrote:
 - /Users/maxchalekson/Desktop/outputs/reg_results_person_year_tidy.csv
 - /Users/maxchalekson/Desktop/outputs/reg_results_coefs_wide.csv
 - /Users/maxchalekson/Desktop/outputs/reg_results_pvals_wide.csv
 - /Users/maxchalekson/Desktop/outputs/reg_model_summaries.txt


## graphs code chunk

In [9]:
# --- Generate role contrast graphs from ALL_person_year.csv ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ---- CONFIG ----
CSV_PATH = Path("/Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/ALL_person_year.csv")
OUT_DIR = Path("/Users/maxchalekson/Desktop/outputs/role_graphs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- LOAD ----
df = pd.read_csv(CSV_PATH)

# Outcomes to plot
p_outcomes = [
    "p_speaking_duration_sec",
    "p_turns",
    "p_interruptions_made",
    "p_overlaps",
    "p_screenshare_segments",
    "p_smile_self_total",
    "p_smile_other_total",
    "p_nods_received",
]
p_outcomes = [c for c in p_outcomes if c in df.columns]

# Contrasts to test
contrasts = ["role_facilitator", "role_on_team", "role_in_funded"]

# Human-readable labels
labels = {
    "role_facilitator": ("Non-facilitator", "Facilitator"),
    "role_on_team": ("Not on team", "On team"),
    "role_in_funded": ("Unfunded team", "Funded team"),
}

# --- Helpers ---
def mean_ci(series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    n = len(s)
    if n == 0:
        return np.nan, np.nan
    mean = s.mean()
    se = s.std(ddof=1) / np.sqrt(n) if n > 1 else 0.0
    ci = 1.96 * se
    return mean, ci

def make_boxplot(df_in, y_col, group_col, title, filename, group_labels):
    d = df_in[[y_col, group_col]].dropna()
    if d.empty: return
    data0 = d.loc[d[group_col] == 0, y_col].values
    data1 = d.loc[d[group_col] == 1, y_col].values
    fig, ax = plt.subplots(figsize=(6, 4.5))
    ax.boxplot([data0, data1], labels=group_labels, showmeans=True)
    ax.set_title(title)
    ax.set_ylabel(y_col)
    ax.set_xlabel(group_col)
    ax.grid(True, linestyle="--", alpha=0.4)
    fig.tight_layout()
    fig.savefig(OUT_DIR / filename, dpi=200)
    plt.close(fig)

def make_bar_meanci(df_in, y_col, group_col, title, filename, group_labels):
    d = df_in[[y_col, group_col]].dropna()
    if d.empty: return
    mean0, ci0 = mean_ci(d.loc[d[group_col] == 0, y_col])
    mean1, ci1 = mean_ci(d.loc[d[group_col] == 1, y_col])
    fig, ax = plt.subplots(figsize=(6, 4.5))
    ax.bar([0, 1], [mean0, mean1], yerr=[ci0, ci1], capsize=6)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(group_labels)
    ax.set_title(title)
    ax.set_ylabel(f"Mean {y_col} (±95% CI)")
    ax.set_xlabel(group_col)
    ax.grid(True, linestyle="--", alpha=0.4, axis="y")
    fig.tight_layout()
    fig.savefig(OUT_DIR / filename, dpi=200)
    plt.close(fig)

def subset_for_contrast(df0, contrast):
    d = df0.copy()
    if contrast == "role_facilitator":
        return d
    elif contrast == "role_on_team":
        if "sessions_total" in d.columns:
            return d[d["sessions_total"].fillna(0) > 0]
        else:
            return d
    elif contrast == "role_in_funded":
        return d[d["role_on_team"] == 1]
    else:
        return d

# --- Generate plots ---
for contrast in contrasts:
    dsub = subset_for_contrast(df, contrast)
    if dsub.empty: continue
    glabels = labels[contrast]

    # Boxplots for speech/turn outcomes
    for y_col in [c for c in ["p_speaking_duration_sec", "p_turns"] if c in p_outcomes]:
        make_boxplot(
            dsub, y_col, contrast,
            f"{y_col} by {contrast}",
            f"box_{y_col}_{contrast}.png",
            group_labels=glabels
        )

    # Bar plots (means ± CI) for counts + affect markers
    for y_col in [c for c in ["p_interruptions_made", "p_overlaps", "p_screenshare_segments",
                              "p_smile_self_total", "p_smile_other_total", "p_nods_received"]
                  if c in p_outcomes]:
        make_bar_meanci(
            dsub, y_col, contrast,
            f"Mean {y_col} (±95% CI) by {contrast}",
            f"bar_{y_col}_{contrast}.png",
            group_labels=glabels
        )

print(f"Graphs saved in: {OUT_DIR}")

/var/folders/17/lphw45ds14n5t74zwjdfybx40000gn/T/ipykernel_75648/1986425863.py:55: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot([data0, data1], labels=group_labels, showmeans=True)
/var/folders/17/lphw45ds14n5t74zwjdfybx40000gn/T/ipykernel_75648/1986425863.py:55: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot([data0, data1], labels=group_labels, showmeans=True)
/var/folders/17/lphw45ds14n5t74zwjdfybx40000gn/T/ipykernel_75648/1986425863.py:55: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot([data0, data1], labels=group_labels, showmeans=True)
/var/folders/17/lphw45ds14n5t74zwjdfybx40000gn

Graphs saved in: /Users/maxchalekson/Desktop/outputs/role_graphs
